# Evaluación Comparativa de Modelos

**Caso de Estudio:** Analítica de Clientes — Predicción de Abandono  
**Asignatura:** SCY1101 — Programación para la Ciencia de Datos  
**Evaluación Parcial N°2**

---

## Descripción

Este notebook realiza la **evaluación comparativa exhaustiva** de todos los modelos supervisados entrenados en `02_supervised_modeling.ipynb`, abarcando:

- **Regresión:** LinearRegression y DecisionTreeRegressor sobre `score_crediticio`.
- **Clasificación:** DecisionTreeClassifier, LogisticRegression y SVC sobre `abandono`.

Para cada modelo se reportan múltiples métricas, se generan visualizaciones avanzadas (matrices de confusión, curvas ROC, análisis de residuales) y se realiza un análisis de **trade-offs** orientado al contexto de negocio.

---

## Requisitos de Software

- `pandas >= 1.1.0`, `numpy >= 2.0.0`
- `scikit-learn >= 1.3`
- `matplotlib >= 3.7.1`, `seaborn >= 0.12.0`

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, mean_absolute_error,
    r2_score, mean_squared_error, confusion_matrix,
    roc_curve, ConfusionMatrixDisplay
)

plt.style.use('bmh')
SEED = 29

In [ ]:
data = pd.read_csv('../data/dataset_clientes.csv')
data.head()

,id_cliente,fecha_registro,edad,genero,region,estado_civil,ingreso_mensual,gasto_mensual,deuda_total,score_crediticio,...,ultima_compra_dias,uso_app,tipo_plan,num_productos,tiene_tarjeta_credito,canal_registro,dia_semana_registro,hora_registro,codigo_postal,abandono
0,1,2021-10-27,66,Otro,Norte,Divorciado,9.243057e+05,524088.303055,2.448145e+06,455.406680,...,356,Bajo,Estandar,3,1,Tienda,Lunes,22,3824,1
1,2,2018-08-25,51,Masculino,Centro,Soltero,1.384687e+06,314259.751474,1.620569e+06,575.048508,...,307,Medio,Premium,4,1,App,Martes,10,4148,0
2,3,2019-05-25,48,Femenino,Norte,Casado,NaN,387192.316142,5.395040e+06,770.716904,...,232,Alto,Premium,4,1,App,Jueves,6,7200,0
3,4,2022-04-20,54,Masculino,Sur,Casado,4.369032e+05,417328.601856,2.999350e+06,442.722671,...,165,Alto,Estandar,2,1,App,Domingo,16,1782,1
4,5,2020-03-19,31,Otro,Centro,Soltero,7.408561e+05,490961.191253,1.637711e+06,468.188403,...,283,Bajo,Estandar,3,1,Web,Martes,8,3448,1


In [ ]:
# Se eliminan los duplicados
data = data.drop_duplicates()

# 1. Preparación de Datos y Reconstrucción de Pipelines

Los mismos pipelines del notebook de modelado se reconstruyen aquí para garantizar reproducibilidad completa. Esto es una práctica estándar cuando los notebooks son independientes; en producción, se usaría `joblib.load()` sobre los modelos serializados.

In [ ]:
class Winsorizer(BaseEstimator, TransformerMixin):
    """
    Tratamiento de atípicos via recorte por percentiles.
    """
    def __init__(self, limits=(0.05, 0.05)):
        self.limits = limits

    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.columns_ = X.columns
        else:
            self.columns_ = np.arange(X.shape[1])
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columns_)
        for col in self.columns_:
            lower = X[col].quantile(self.limits[0])
            upper = X[col].quantile(1 - self.limits[1])
            X = X.astype('float64')
            X[col] = np.clip(X[col], lower, upper)
        return X

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.array(self.columns_)
        return np.array(input_features)


def tratar_duplicados(X: pd.DataFrame, drop: bool = True) -> pd.DataFrame:
    """
    Tratamiento de duplicados.
    Si drop=True elimina filas duplicadas, si no las deja.
    """
    return X.drop_duplicates() if drop else X


class CorrelationFilter(BaseEstimator, TransformerMixin):
    """
    Elimina variables con alta correlación (multicolinealidad).
    """
    def __init__(self, threshold=0.9):
        self.threshold = threshold
        self.columns_to_drop_ = None

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        corr_matrix = X_df.corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        self.columns_to_drop_ = [
            col for col in upper.columns if any(upper[col] > self.threshold)
        ]
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)
        return X_df.drop(columns=self.columns_to_drop_, errors='ignore').values


class DataFrameConverter(BaseEstimator, TransformerMixin):
    """
    Convierte el array de ColumnTransformer en DataFrame con nombres de columnas.
    """
    def __init__(self, preprocessor):
        self.preprocessor = preprocessor
        self.feature_names_ = None

    def fit(self, X, y=None):
        self.feature_names_ = self.preprocessor.get_feature_names_out()
        return self

    def transform(self, X):
        return pd.DataFrame(X, columns=self.feature_names_)

In [ ]:
def evaluar(modelo, X_train, X_test, y_train, y_test):
    """
    Entrena el modelo y retorna métricas de clasificación.
    """
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]
    return {
        'accuracy':  accuracy_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall':    recall_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_prob)
    }

# 2. Evaluación de Modelos de Regresión

## 2.1 Marco de Evaluación para Regresión

Se evalúan los modelos de regresión sobre el conjunto de prueba con tres métricas complementarias:

| Métrica | Fórmula | Interpretación en contexto |
|---|---|---|
| **R²** | 1 - SS_res/SS_tot | Proporción de varianza del score crediticio explicada por el modelo. R²=0 ≡ predecir siempre la media |
| **MAE** | mean(\|y - ŷ\|) | Error promedio en puntos de score. Ej: MAE=79 significa error promedio de 79 puntos |
| **RMSE** | √mean((y - ŷ)²) | Penaliza errores grandes. Si RMSE >> MAE, hay errores muy grandes ocasionales |

> **Variable objetivo:** `score_crediticio` (μ ≈ 600, σ ≈ 100)

In [ ]:
target_reg = 'score_crediticio'

features_num = [
    'edad', 'ingreso_mensual', 'gasto_mensual', 'deuda_total',
    'antiguedad_meses', 'frecuencia_compra', 'ultima_compra_dias', 'num_productos'
]
features_cat = [
    'genero', 'region', 'estado_civil', 'uso_app', 'tipo_plan', 'canal_registro'
]

X_reg = data[features_num + features_cat]
y_reg = data[target_reg]

mask = y_reg.notna()
X_reg, y_reg = X_reg[mask], y_reg[mask]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=44
)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('winsorizer', Winsorizer()),
    ('imputer',    SimpleImputer(strategy='mean')),
    ('scaler',     StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, features_num),
        ('cat', categorical_transformer, features_cat)
    ],
    remainder='drop',
    force_int_remainder_cols=False
)

In [ ]:
preprocessor_lr = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, features_num),
        ('cat', categorical_transformer, features_cat)
    ],
    remainder='drop',
    force_int_remainder_cols=False
)

modelo_lr = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor_lr),
    ('conversion',    DataFrameConverter(preprocessor_lr)),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        LinearRegression())
])

modelo_dtr = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('conversion',    DataFrameConverter(preprocessor)),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        DecisionTreeRegressor(
                          max_depth=2, min_samples_leaf=200,
                          min_samples_split=100, random_state=44))
])

modelo_lr.fit(X_train_reg, y_train_reg)
modelo_dtr.fit(X_train_reg, y_train_reg)

y_pred_lr  = modelo_lr.predict(X_test_reg)
y_pred_dtr = modelo_dtr.predict(X_test_reg)

resultados_reg = pd.DataFrame({
    'Modelo': ['LinearRegression', 'DecisionTreeRegressor'],
    'R2':   [r2_score(y_test_reg, y_pred_lr),   r2_score(y_test_reg, y_pred_dtr)],
    'MAE':  [mean_absolute_error(y_test_reg, y_pred_lr),  mean_absolute_error(y_test_reg, y_pred_dtr)],
    'RMSE': [np.sqrt(mean_squared_error(y_test_reg, y_pred_lr)),
             np.sqrt(mean_squared_error(y_test_reg, y_pred_dtr))]
})

resultados_reg.set_index('Modelo', inplace=True)
print(resultados_reg.to_string())

                             R2        MAE       RMSE
Modelo                                               
LinearRegression       0.000856  79.446319  98.917356
DecisionTreeRegressor  0.000761  79.454921  98.922087


## 2.2 Interpretación de Resultados de Regresión

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (nombre, y_pred) in zip(axes, [('LinearRegression', y_pred_lr), ('DecisionTreeRegressor', y_pred_dtr)]):
    ax.scatter(y_test_reg, y_pred, alpha=0.3, s=8, color='steelblue')
    lims = [y_test_reg.min(), y_test_reg.max()]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Predicción perfecta')
    ax.set_xlabel('Valor real (score_crediticio)', fontsize=10)
    ax.set_ylabel('Valor predicho', fontsize=10)
    ax.set_title(f'{nombre}\nPredichos vs. Reales', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Análisis de Predicciones — Modelos de Regresión', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/03_regresion_pred_vs_real.png', dpi=150, bbox_inches='tight')
plt.show()

# Gráfico de residuales
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (nombre, y_pred) in zip(axes, [('LinearRegression', y_pred_lr), ('DecisionTreeRegressor', y_pred_dtr)]):
    residuales = y_test_reg - y_pred
    ax.scatter(y_pred, residuales, alpha=0.3, s=8, color='coral')
    ax.axhline(0, color='black', linewidth=1.2, linestyle='--')
    ax.set_xlabel('Valor predicho', fontsize=10)
    ax.set_ylabel('Residual (real − predicho)', fontsize=10)
    ax.set_title(f'{nombre}\nGráfico de Residuales', fontsize=11, fontweight='bold')

plt.suptitle('Análisis de Residuales — Modelos de Regresión', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/03_regresion_residuales.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretación de los resultados de regresión

**R² ≈ 0.001 en ambos modelos** indica que los features disponibles explican menos del 0.1% de la varianza de `score_crediticio`. Esto es coherente con las correlaciones numéricas muy bajas detectadas en el EDA (ninguna variable supera r = 0.21 con el score crediticio).

**Interpretación del MAE ≈ 79 puntos:** El score crediticio varía entre ~220 y ~998 (rango ≈ 778 puntos). Un error medio de 79 puntos representa aproximadamente un **10% del rango total**, lo cual es significativo. Ambos modelos tienen un desempeño similar al de predecir siempre la media del score (benchmark trivial).

**Gráficos de residuales:** La distribución de residuales sin patrón sistemático visible (residuales centrados en 0, sin heterocedasticidad evidente) indica que el problema no es un error de especificación del modelo, sino que las features disponibles **no contienen información suficiente** para predecir el score crediticio. Una mejora requeriría variables adicionales (historial de pagos, tipo de crédito, etc.).

**Conclusión de negocio:** Los modelos de regresión actuales no son adecuados para uso en producción. Se recomienda enriquecer el dataset con variables financieras adicionales o explorar modelos de ensamble (Random Forest, Gradient Boosting) que puedan capturar interacciones más complejas.

# 3. Evaluación de Modelos de Clasificación

## 3.1 Marco de Evaluación para Clasificación

Para clasificación binaria en el contexto del abandono de clientes, se utilizan cinco métricas complementarias:

| Métrica | Definición | Relevancia para churn |
|---|---|---|
| **Accuracy** | (TP+TN)/(TP+TN+FP+FN) | Poco informativa con clases desbalanceadas |
| **Precision** | TP/(TP+FP) | ¿De los clientes que predigo que abandonan, cuántos realmente abandonan? Alta precisión = menos falsos alarmas |
| **Recall** | TP/(TP+FN) | ¿De los clientes que realmente abandonan, cuántos detecto? **Métrica prioritaria**: no perder clientes en riesgo |
| **F1-Score** | 2·(P·R)/(P+R) | Media armónica entre Precision y Recall; penaliza modelos extremos |
| **ROC-AUC** | Área bajo la curva ROC | Capacidad discriminativa del modelo independiente del umbral; AUC=0.5 ≡ clasificador aleatorio |

**Trade-off central — Precision vs. Recall:**
> Subir el Recall implica capturar más clientes en riesgo real, pero también generar más falsas alarmas (menor Precision). En este contexto, **es preferible una Precision baja a un Recall bajo**, ya que el costo de llamar a un cliente que no iba a abandonar (FP) es mucho menor que el de ignorar a uno que sí iba a hacerlo (FN).

In [ ]:
target_cls = 'abandono'

X_cls = data[features_num + features_cat]
y_cls = data[target_cls]

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=29, stratify=y_cls
)

In [ ]:
pipeline_dtc = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        DecisionTreeClassifier(random_state=29))
])

pipeline_logreg = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        LogisticRegression(class_weight='balanced', max_iter=10000, random_state=29))
])

numeric_transformer_svm = Pipeline(steps=[
    ('winsorizer', Winsorizer()),
    ('imputer',    SimpleImputer(strategy='mean')),
    ('scaler',     StandardScaler())
])

preprocessor_svm = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_svm, features_num),
        ('cat', categorical_transformer,  features_cat)
    ],
    remainder='drop',
    force_int_remainder_cols=False
)

pipeline_svm = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor_svm),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=29))
])

metricas_dtc    = evaluar(pipeline_dtc,    X_train_cls, X_test_cls, y_train_cls, y_test_cls)
metricas_logreg = evaluar(pipeline_logreg, X_train_cls, X_test_cls, y_train_cls, y_test_cls)
metricas_svm    = evaluar(pipeline_svm,    X_train_cls, X_test_cls, y_train_cls, y_test_cls)

resultados_cls = pd.DataFrame({
    'Modelo':    ['DecisionTreeClassifier', 'LogisticRegression', 'SVM'],
    'Accuracy':  [metricas_dtc['accuracy'],  metricas_logreg['accuracy'],  metricas_svm['accuracy']],
    'F1':        [metricas_dtc['f1'],         metricas_logreg['f1'],        metricas_svm['f1']],
    'Precision': [metricas_dtc['precision'],  metricas_logreg['precision'], metricas_svm['precision']],
    'Recall':    [metricas_dtc['recall'],     metricas_logreg['recall'],    metricas_svm['recall']],
    'ROC AUC':   [metricas_dtc['roc_auc'],    metricas_logreg['roc_auc'],   metricas_svm['roc_auc']]
})

resultados_cls.set_index('Modelo', inplace=True)
print(resultados_cls.to_string())

In [ ]:
# ── Visualización 1: Gráfico comparativo de barras por métrica ──────────────
metricas_labels = ['Accuracy', 'F1', 'Precision', 'Recall', 'ROC AUC']
modelos_nombres = ['DecisionTree', 'LogisticReg', 'SVM']
colores = ['#4C72B0', '#DD8452', '#55A868']

valores = np.array([
    [metricas_dtc['accuracy'],    metricas_dtc['f1'],    metricas_dtc['precision'],    metricas_dtc['recall'],    metricas_dtc['roc_auc']],
    [metricas_logreg['accuracy'], metricas_logreg['f1'], metricas_logreg['precision'], metricas_logreg['recall'], metricas_logreg['roc_auc']],
    [metricas_svm['accuracy'],    metricas_svm['f1'],    metricas_svm['precision'],    metricas_svm['recall'],    metricas_svm['roc_auc']],
])

x = np.arange(len(metricas_labels))
width = 0.25
fig, ax = plt.subplots(figsize=(14, 6))
for i, (modelo, color) in enumerate(zip(modelos_nombres, colores)):
    bars = ax.bar(x + i * width, valores[i], width, label=modelo, color=color, alpha=0.85)
    for bar, val in zip(bars, valores[i]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(metricas_labels, fontsize=11)
ax.set_ylim(0, 0.85)
ax.set_ylabel('Valor de la métrica', fontsize=11)
ax.set_title('Comparación de Métricas por Modelo de Clasificación\n(Conjunto de Prueba)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(0.5, color='red', linestyle='--', linewidth=0.8, alpha=0.5, label='Referencia 0.5')
plt.tight_layout()
plt.savefig('../results/plots/03_comparacion_metricas.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Visualización 2: Matrices de Confusión ────────────────────────────────────
modelos_dict = {
    'DecisionTreeClassifier': pipeline_dtc,
    'LogisticRegression': pipeline_logreg,
    'SVM (RBF)': pipeline_svm
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (nombre, pipeline) in zip(axes, modelos_dict.items()):
    y_pred = pipeline.predict(X_test_cls)
    cm = confusion_matrix(y_test_cls, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Permanece (0)', 'Abandona (1)'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(nombre, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicho', fontsize=9)
    ax.set_ylabel('Real', fontsize=9)

plt.suptitle('Matrices de Confusión — Modelos de Clasificación\n(Conjunto de Prueba)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/03_matrices_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Visualización 3: Curvas ROC comparativas ──────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
colores = ['#4C72B0', '#DD8452', '#55A868']

for (nombre, pipeline), color in zip(modelos_dict.items(), colores):
    y_prob = pipeline.predict_proba(X_test_cls)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_cls, y_prob)
    auc = roc_auc_score(y_test_cls, y_prob)
    ax.plot(fpr, tpr, label=f'{nombre} (AUC = {auc:.3f})', color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Clasificador aleatorio (AUC = 0.500)')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
ax.set_xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12)
ax.set_ylabel('Tasa de Verdaderos Positivos (Recall)', fontsize=12)
ax.set_title('Curvas ROC Comparativas\nPredicción de Abandono de Clientes', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig('../results/plots/03_curvas_roc.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.2 Interpretación Detallada de Resultados de Clasificación

### Análisis de las Matrices de Confusión

La matriz de confusión desglosa las predicciones en cuatro categorías críticas para el negocio:

- **Verdaderos Positivos (TP):** Clientes que abandonan y el modelo predice correctamente. → Acción de retención activada correctamente.
- **Falsos Negativos (FN):** Clientes que abandonan y el modelo NO detecta. → **El error más costoso**: se pierde el cliente sin intervención.
- **Falsos Positivos (FP):** Clientes que NO abandonan pero el modelo predice abandono. → Costo de campaña de retención innecesaria.
- **Verdaderos Negativos (TN):** Clientes que permanecen y el modelo predice correctamente. → No requiere acción.

### Análisis de las Curvas ROC

La curva ROC muestra el trade-off entre **Recall (sensibilidad)** y **FPR (1 - especificidad)** a distintos umbrales de clasificación:

- **LogisticRegression (AUC ≈ 0.68):** Mejor capacidad discriminativa general. Curva más alejada de la diagonal.
- **SVM (AUC ≈ 0.65):** Segundo mejor, marginalmente inferior a LogReg.
- **DecisionTreeClassifier (AUC ≈ 0.54):** Cercano al clasificador aleatorio; el árbol sin regularización memoriza el entrenamiento y generaliza mal.

### Comparación y Selección del Mejor Modelo Baseline

Todos los modelos usan `class_weight='balanced'` (excepto DecisionTree en esta fase baseline), lo que prioriza la detección de la clase minoritaria (abandono).

| Criterio | Ganador | Justificación |
|---|---|---|
| Recall (detectar abandono) | SVM | Mayor Recall (~0.63) gracias a `class_weight='balanced'` y kernel RBF |
| ROC-AUC (discriminación) | LogisticRegression | AUC ≈ 0.68, mejor capacidad de separación de clases |
| Interpretabilidad | LogisticRegression | Coeficientes expresan log-odds de abandono por variable |
| Equilibrio F1 | LogisticRegression | Mejor balance Precision-Recall |

> **Conclusión:** Para el problema de churn, `LogisticRegression` con `class_weight='balanced'` es el mejor modelo baseline, combinando capacidad discriminativa (AUC ≈ 0.68), Recall ~0.62 e interpretabilidad para el equipo de negocio. La optimización de hiperparámetros buscará mejorar estas métricas en el notebook `04_hyperparameter_optimization.ipynb`.